In [3]:
import pandas as pd
import numpy as np
import os

In [4]:
# 1. Đọc dữ liệu gốc (Sử dụng trực tiếp dataset.csv)
# Lưu ý: Dữ liệu gốc thường dùng dấu phẩy ',' hoặc chấm phẩy ';'. Pandas thường tự nhận diện tốt với sep=','.
df = pd.read_csv('dataset/dataset.csv')
print("Kích thước ban đầu:", df.shape)
print("\nMột số cột gốc trong dataset:\n", df.columns.tolist()[:10])

Kích thước ban đầu: (4424, 35)

Một số cột gốc trong dataset:
 ['Marital status', 'Application mode', 'Application order', 'Course', 'Daytime/evening attendance', 'Previous qualification', 'Nacionality', "Mother's qualification", "Father's qualification", "Mother's occupation"]


In [5]:
# 2. Đổi tên cột từ định dạng gốc sang định dạng của hệ thống DSS
rename_dict = {
    'Curricular units 1st sem (enrolled)': '1st_sem_enrolled',
    'Curricular units 1st sem (evaluations)': '1st_sem_evaluations',
    'Curricular units 1st sem (without evaluations)': '1st_sem_without_evaluations',
    'Curricular units 1st sem (approved)': '1st_sem_approved',
    'Curricular units 1st sem (grade)': '1st_sem_grade'
}

df = df.rename(columns=rename_dict)
print("\nĐã đổi tên cột thành công!")


Đã đổi tên cột thành công!


In [6]:
# 3. Kỹ thuật đặc trưng (Feature Engineering)

# Tạo 1st_sem_failed: Do dataset gốc không có cột failed, 
# ta tính bằng số môn đã đánh giá (thi) TRỪ ĐI số môn đạt (tránh dùng enrolled - approved vì có những môn bỏ thi)
df['1st_sem_failed'] = df['1st_sem_evaluations'] - df['1st_sem_approved']
# Đảm bảo không có giá trị âm (trường hợp bất thường về dữ liệu)
df['1st_sem_failed'] = df['1st_sem_failed'].clip(lower=0)

# Tạo 1st_sem_pass_rate (Ngăn lỗi ZeroDivisionError)
df['1st_sem_pass_rate'] = np.where(
    df['1st_sem_evaluations'] == 0, 
    0, 
    df['1st_sem_approved'] / df['1st_sem_evaluations']
)

In [7]:
# 4. Chuẩn hóa nhãn (Target)
target_mapping = {
    'Graduate': 'Safe',
    'Enrolled': 'Caution',
    'Dropout': 'Dropout'
}
df['Target'] = df['Target'].map(target_mapping)
print("\nPhân bố nhãn sau chuẩn hóa:\n", df['Target'].value_counts())


Phân bố nhãn sau chuẩn hóa:
 Target
Safe       2209
Dropout    1421
Caution     794
Name: count, dtype: int64


In [8]:
# 5. Lọc đúng 7 đặc trưng + Target để xuất ra file
feature_cols = [
    '1st_sem_enrolled',
    '1st_sem_evaluations',
    '1st_sem_without_evaluations',
    '1st_sem_approved',
    '1st_sem_failed',
    '1st_sem_grade',
    '1st_sem_pass_rate',
    'Target'
]
df_processed = df[feature_cols]

# 6. Lưu ra file CSV mới phục vụ cho việc train model (Lưu vào thư mục dataset/)
os.makedirs('dataset', exist_ok=True)
df_processed.to_csv('dataset/dataset_preprocessed.csv', index=False)
print("\nĐã lưu dữ liệu tiền xử lý thành công vào dataset/dataset_preprocessed.csv!")
df_processed.head()


Đã lưu dữ liệu tiền xử lý thành công vào dataset/dataset_preprocessed.csv!


,1st_sem_enrolled,1st_sem_evaluations,1st_sem_without_evaluations,1st_sem_approved,1st_sem_failed,1st_sem_grade,1st_sem_pass_rate,Target
0,0,0,0,0,0,0.000000,0.000000,Dropout
1,6,6,0,6,0,14.000000,1.000000,Safe
2,6,0,0,0,0,0.000000,0.000000,Dropout
3,6,8,0,6,2,13.428571,0.750000,Safe
4,6,9,0,5,4,12.333333,0.555556,Safe
